# Examen Final – Sistema RAG sobre arXiv Paper Abstracts
**Desarrollado por:** Erick Romero

En este notebook se detalla el diseño, implementación y evaluación de un sistema de Recuperación Aumentada por Generación (RAG).

## A. Preparación del corpus
Cargarmos la base de datos de Kaggle, seleccionaremos las columnas relevantes

In [1]:
import kagglehub

# Descargar la última versión del dataset
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'arxiv-paper-abstracts' dataset.
Path to dataset files: /kaggle/input/arxiv-paper-abstracts


In [2]:
import os

os.listdir(path)

['arxiv_data_210930-054931.csv', 'arxiv_data.csv']

In [3]:
import pandas as pd
import os

csv_path = os.path.join(path, "arxiv_data.csv")

df = pd.read_csv(csv_path)
print(f"Número de documentos: {len(df):,}")
df.head()

Número de documentos: 51,774


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51774 entries, 0 to 51773
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   titles     51774 non-null  object
 1   summaries  51774 non-null  object
 2   terms      51774 non-null  object
dtypes: object(3)
memory usage: 1.2+ MB


In [5]:
df.columns

Index(['titles', 'summaries', 'terms'], dtype='object')

### Limpieza de datos
Eliminamos registros nulos y duplicados, y normalizamos espacios y caracteres en los textos.

In [6]:
# Eliminacion de duplicados
duplicados = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicados eliminados: {duplicados}")

# Eliminacion de nulos
nulos_antes = df.isnull().sum().sum()
df = df.dropna()
print(f"Filas con nulos eliminadas: {nulos_antes}")

# Limpieza de espacios y caracteres especiales
for col in ['titles', 'summaries']:
    df[col] = df[col].str.strip()
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    df[col] = df[col].str.replace(r'[^\w\s\.\,\-\:\;\\(\\)\[\]\/\?\!\'\']', '', regex=True)

# Comprobamos
print(f"\nDocumentos después de limpieza: {len(df):,}")
print(f"\nNulos por columna:\n{df.isnull().sum()}")
print(f"\nDuplicados restantes: {df.duplicated().sum()}")
print(f"\nNúmero de documentos finales: {len(df):,}")
df.head()

Duplicados eliminados: 12783
Filas con nulos eliminadas: 0

Documentos después de limpieza: 38,991

Nulos por columna:
titles       0
summaries    0
terms        0
dtype: int64

Duplicados restantes: 0

Número de documentos finales: 38,991


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"


In [7]:
df["document"] = (
    "Title: " + df["titles"] +
    "\n\nSummaries: " + df["summaries"]
)

In [8]:
df.head()

,titles,summaries,terms,document
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']",Title: Survey on Semantic Stereo Matching / Se...
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']",Title: FUTURE-AI: Guiding Principles and Conse...
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']",Title: Enforcing Mutual Consistency of Hard Re...
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV'],Title: Parameter Decoupling Strategy for Semi-...
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']",Title: Background-Foreground Segmentation for ...


# B. Representación mediante Embeddings

Para representar semánticamente los documentos del corpus se empleó un modelo de **Sentence Transformers**.

Se seleccionó el modelo **BAAI/bge-small-en-v1.5**, ya que ofrece un buen equilibrio entre calidad, velocidad y tamaño del modelo. Además, está optimizado para tareas de búsqueda semántica (semantic search), siendo ampliamente utilizado en sistemas RAG.

Cada documento (compuesto por el título y el abstract) es transformado en un vector denso de alta dimensión. Estos vectores permiten medir la similitud semántica entre consultas y documentos utilizando métricas como la similitud del coseno.

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
# Calcula los embeddings y los guarda
embeddings = embedding_model.encode(
    df["document"].tolist(),
    batch_size=120,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
# Guardar embeddings y corpus limpio
np.save('arxiv_embeddings.npy', embeddings)
df.to_csv('arxiv_cleaned.csv', index=False)
print(f'Embeddings creados: {embeddings.shape}. Guardados en arxiv_embeddings.npy')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/325 [00:00<?, ?it/s]

Embeddings creados: (38991, 384). Guardados en arxiv_embeddings.npy


In [10]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(38991, 384)


# C. Almacenamiento y búsqueda vectorial

Los embeddings generados en la etapa anterior se almacenan en una base de datos vectorial utilizando ****.

Cada documento se almacena junto con:

- Un identificador único.
- El embedding correspondiente.
- El texto completo del documento.
- Metadatos (título y categoría).

Esto permite realizar búsquedas semánticas eficientes mediante la comparación entre el embedding de una consulta y los embeddings almacenados.

In [12]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 MB 7.5 MB/s eta 0:00:00:00:0100:01


In [13]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(
    embeddings.astype("float32")
)

print(index.ntotal)

38991


In [14]:
faiss.write_index(
    index,
    "arxiv_index.faiss"
)

Cada embedding en el índice FAISS mantiene el mismo orden que el DataFrame `df`. Esto nos permite, al recuperar los índices de los vecinos más cercanos, acceder directamente a los documentos correspondientes.

In [15]:
# Función de búsqueda semántica
def buscar(query, k=5):
    # 1. Generar embedding de la consulta (normalizado)
    q_emb = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    # 2. Buscar los k vecinos más cercanos
    scores, indices = index.search(q_emb, k)
    # 3. Recuperar documentos del DataFrame
    resultados = df.iloc[indices[0]].copy()
    resultados["score"] = scores[0]
    return resultados[["titles", "terms", "score", "document"]]

# Ejemplo de búsqueda
query_ejemplo = "What are the main applications of Graph Neural Networks?"
resultados = buscar(query_ejemplo, k=3)
print(f"Consulta: {query_ejemplo}\n")
resultados

Consulta: What are the main applications of Graph Neural Networks?



,titles,terms,score,document
19175,"Graph Neural Networks: Methods, Applications, ...","['cs.LG', 'cs.AI', '68Txx', 'I.2.6; I.2; I.5']",0.836709,"Title: Graph Neural Networks: Methods, Applica..."
3066,On Node Features for Graph Neural Networks,"['cs.LG', 'stat.ML']",0.819328,Title: On Node Features for Graph Neural Netwo...
22725,"Graph Neural Networks: Architectures, Stabilit...","['cs.LG', 'stat.ML']",0.816302,"Title: Graph Neural Networks: Architectures, S..."


# D. Recuperación

Para recuperar documentos relevantes se implementa un enfoque híbrido:

1. Recuperación inicial mediante búsqueda vectorial con FAISS.
2. Re-ranking mediante un modelo Cross Encoder.

FAISS permite encontrar rápidamente los documentos cuyos embeddings tienen mayor similitud semántica con la consulta.

Posteriormente, el Cross Encoder evalúa directamente la relación entre la consulta y cada documento recuperado, mejorando la precisión de los resultados.